In [ ]:
import nb_setup
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from src.data import load_tokenised_data
from src.answerability.evaluation import evaluate_answerability_predictions
from src.answerability.gru_dataset import PretrainedAnswerabilityDataset, make_collate_fn
from src.answerability.logistic_regression import (
    train_logistic_regression,
    predict_logistic_regression,
    logistic_feature_weights,
)
from src.answerability.gru_model import (
    PretrainedGRUClassifier,
    train_one_epoch_answerability,
    predict_answerability,
    evaluate_macro_f1
)
from src.utils import enforce_reproducibility
from src.rule_based import rule_based_classifier_batch
from src.translator import NLLBTranslator
translator = NLLBTranslator()

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

In [2]:
enforce_reproducibility()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [3]:
df_train_tokenised, df_val_tokenised = load_tokenised_data()

In [4]:
lr_model, lr_vectorizer = train_logistic_regression(df_train_tokenised)

In [5]:
lr_pred, lr_prob = predict_logistic_regression(lr_model, lr_vectorizer, df_val_tokenised)

In [6]:
y_val = df_val_tokenised["answerable"].astype(int).to_numpy()
val_langs = df_val_tokenised["lang"].to_numpy()

In [7]:
lr_results = evaluate_answerability_predictions(y_val, lr_pred, val_langs, "Logistic Regression").round(4)
lr_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Logistic Regression,Overall,1155,0.7236,0.8108,0.3659,0.5042,0.9038,0.9859,0.9431
1,Logistic Regression,Arabic,415,0.9459,0.8889,0.9231,0.9057,0.9889,0.9835,0.9862
2,Logistic Regression,Korean,356,0.4863,0.0000,0.0000,0.0000,0.9466,1.0000,0.9726
3,Logistic Regression,Telugu,384,0.5383,0.6000,0.1290,0.2124,0.7775,0.9725,0.8641


In [8]:
feature_weights = logistic_feature_weights(lr_model, lr_vectorizer)

In [9]:
feature_weights.head(15)

,feature,weight
0,q_word=هل,-6.758095
1,q_word=수,-1.216855
2,q_word=있을까,-0.925031
3,q_word=있는가,-0.923192
4,c_word=center,-0.884653
5,q_word=దేశం,-0.881877
6,c_word=armed,-0.799558
7,c_word=elections,-0.761923
8,q_word=ريشتر,-0.746954
9,q_word=الجيولوجي,-0.746954


In [10]:
feature_weights.tail(15)

,feature,weight
56966,q_word=무엇인가요,0.545076
56967,c_word=official,0.564311
56968,q_word=أين,0.567153
56969,c_word=following,0.586641
56970,q_word=어디인가,0.628492
56971,c_word=so,0.631174
56972,c_word=.,0.661460
56973,q_word=ఎన్ని,0.672766
56974,q_word=كم,0.713128
56975,context_has_digit,0.758185


In [11]:
majority_class = df_train_tokenised["answerable"].astype(int).mode()[0]
majority_pred = np.full(len(y_val), majority_class)

majority_results = evaluate_answerability_predictions(y_val, majority_pred, val_langs, "Majority Baseline").round(4)
majority_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Majority Baseline,Overall,1155,0.4618,0.0,0.0,0.0,0.8580,1.0,0.9236
1,Majority Baseline,Arabic,415,0.4666,0.0,0.0,0.0,0.8747,1.0,0.9332
2,Majority Baseline,Korean,356,0.4863,0.0,0.0,0.0,0.9466,1.0,0.9726
3,Majority Baseline,Telugu,384,0.4311,0.0,0.0,0.0,0.7578,1.0,0.8622


In [12]:
rule_pred = np.array(
    rule_based_classifier_batch(
        df_val_tokenised,
        translator,
        threshold=0.3,
        batch_size=8,
    ),
    dtype=int,
)

rule_results = evaluate_answerability_predictions(
    y_val, rule_pred, val_langs, "Rule-Based"
).round(4)

rule_results

Translating ar:   0%|          | 0/52 [00:00<?, ?it/s]

Translating ko:   0%|          | 0/45 [00:00<?, ?it/s]

Translating te:   0%|          | 0/48 [00:00<?, ?it/s]

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Rule-Based,Overall,1155,0.5690,0.2409,0.4024,0.3014,0.8888,0.7901,0.8365
1,Rule-Based,Arabic,415,0.5296,0.1765,0.3462,0.2338,0.8914,0.7686,0.8254
2,Rule-Based,Korean,356,0.4569,0.0460,0.2105,0.0755,0.9442,0.7537,0.8383
3,Rule-Based,Telugu,384,0.6709,0.5176,0.4731,0.4944,0.8361,0.8591,0.8475


In [13]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
xlmr = AutoModel.from_pretrained(MODEL_NAME)

pretrained_embeddings = (
    xlmr
    .get_input_embeddings()
    .weight
    .detach()
    .cpu()
    .clone()
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
train_dataset = PretrainedAnswerabilityDataset(df_train_tokenised, tokenizer)
val_dataset = PretrainedAnswerabilityDataset(df_val_tokenised, tokenizer)

collate_fn = make_collate_fn(tokenizer.pad_token_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
)

In [15]:
model = PretrainedGRUClassifier(
    pretrained_embeddings=pretrained_embeddings,
    hidden_dim=64,
    dropout_prob=0.3,
    padding_idx=tokenizer.pad_token_id,
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters(),
    ),
    lr=1e-3,
    weight_decay=1e-4,
)

In [16]:
NUM_EPOCHS = 10
PATIENCE = 3

best_val_f1 = -1
best_model_state = None
best_epoch = None

epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):

    train_loss = (
        train_one_epoch_answerability(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )
    )

    val_f1 = evaluate_macro_f1(model, val_loader, device)

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1

        best_model_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

Epoch 1/10 | Train loss: 0.1691 | Val Macro F1: 0.6986
Epoch 2/10 | Train loss: 0.0930 | Val Macro F1: 0.6986
Epoch 3/10 | Train loss: 0.0773 | Val Macro F1: 0.8046
Epoch 4/10 | Train loss: 0.0764 | Val Macro F1: 0.8484
Epoch 5/10 | Train loss: 0.0700 | Val Macro F1: 0.8612
Epoch 6/10 | Train loss: 0.0609 | Val Macro F1: 0.8023
Epoch 7/10 | Train loss: 0.0503 | Val Macro F1: 0.8137
Epoch 8/10 | Train loss: 0.0420 | Val Macro F1: 0.8252
Early stopping


In [17]:
print(f"Best epoch: {best_epoch}")
print(f"Best validation Macro F1: {best_val_f1:.4f}")

Best epoch: 5
Best validation Macro F1: 0.8612


In [18]:
model.load_state_dict(best_model_state)
model.to(device)

PretrainedGRUClassifier(
  (embeddings): Embedding(250002, 768, padding_idx=1)
  (gru): GRU(768, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)

In [19]:
gru_pred, gru_prob, y_val, val_langs = predict_answerability(model, val_loader, device)
gru_results = evaluate_answerability_predictions(y_val, gru_pred, val_langs, "Pretrained GRU").round(4)
gru_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Pretrained GRU,Overall,1155,0.8612,0.8790,0.6646,0.7569,0.9467,0.9849,0.9654
1,Pretrained GRU,Arabic,415,0.9517,0.8909,0.9423,0.9159,0.9917,0.9835,0.9876
2,Pretrained GRU,Korean,356,0.9002,0.8333,0.7895,0.8108,0.9882,0.9911,0.9896
3,Pretrained GRU,Telugu,384,0.7692,0.8824,0.4839,0.6250,0.8559,0.9794,0.9135


In [20]:
errors_df = df_val_tokenised.copy()
errors_df["true"] = y_val
errors_df["pred"] = gru_pred
errors_df["error_type"] = None
errors_df.loc[(errors_df["true"] == 1) & (errors_df["pred"] == 0), "error_type"] = "false_negative"
errors_df.loc[(errors_df["true"] == 0) & (errors_df["pred"] == 1), "error_type"] = "false_positive"
errors_only = errors_df[errors_df["error_type"].notna()]

pd.crosstab(errors_only["lang"], errors_only["error_type"])

error_type,false_negative,false_positive
lang,,
ar,6,3
ko,3,4
te,6,48


In [21]:
all_results = pd.concat([majority_results, rule_results, lr_results, gru_results], ignore_index=True).round(4)
all_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Majority Baseline,Overall,1155,0.4618,0.0000,0.0000,0.0000,0.8580,1.0000,0.9236
1,Majority Baseline,Arabic,415,0.4666,0.0000,0.0000,0.0000,0.8747,1.0000,0.9332
2,Majority Baseline,Korean,356,0.4863,0.0000,0.0000,0.0000,0.9466,1.0000,0.9726
3,Majority Baseline,Telugu,384,0.4311,0.0000,0.0000,0.0000,0.7578,1.0000,0.8622
4,Rule-Based,Overall,1155,0.5690,0.2409,0.4024,0.3014,0.8888,0.7901,0.8365
5,Rule-Based,Arabic,415,0.5296,0.1765,0.3462,0.2338,0.8914,0.7686,0.8254
6,Rule-Based,Korean,356,0.4569,0.0460,0.2105,0.0755,0.9442,0.7537,0.8383
7,Rule-Based,Telugu,384,0.6709,0.5176,0.4731,0.4944,0.8361,0.8591,0.8475
8,Logistic Regression,Overall,1155,0.7236,0.8108,0.3659,0.5042,0.9038,0.9859,0.9431
9,Logistic Regression,Arabic,415,0.9459,0.8889,0.9231,0.9057,0.9889,0.9835,0.9862


In [22]:
sample_errors = (
    errors_only
    .groupby(["lang", "error_type"], group_keys=False)
    .head(2)
)

sample_errors[
    ["lang", "error_type", "question", "context"]
]

,lang,error_type,question,context
35,ar,false_negative,هل مصرهي دولة نامية؟,"In recent years, new terms have emerged to des..."
112,ar,false_negative,هل يعتبر بولس من حواري المسيح؟,"Paul the Apostle (; ; ; c. 5 – c. 64 or 67), c..."
191,ar,false_positive,لماذا تم تسمية نبتة القديس يوحنا المثقبة بهذا ...,"The specific epithet ""perforatum"" is Latin, re..."
341,ar,false_positive,ماكان سبب إندلاع الحرب الأهلية الأنغولية ؟,Möngke Khan's brother Hulagu Khan broke off hi...
468,ko,false_positive,미국은 주마다 법이 다른가?,"In the United States, state law refers to the ..."
520,ko,false_positive,2019년 현재 페터 베르크만 사건은 해결되었는가?,The Peter Bergmann case is an unsolved mystery...
569,ko,false_negative,인도네시아에서는 동성결혼이 합법적인가?,Indonesian law does not recognise same-sex mar...
644,ko,false_negative,인도네시아에서는 동성결혼이 합법적인가?,"Lesbian, gay, bisexual, and transgender (LGBT)..."
814,te,false_negative,అత్యంత శక్తివంతమైన నావికా రక్షణ దళం ఏ దేశానికి...,The United States Navy (USN) is the naval warf...
867,te,false_negative,లోకోమోటివ్ బాయిలరును మొదట ఏ దేశంలో ప్రవేశపెట్ట...,A steam locomotive is a locomotive whose prima...


In [23]:

category_map = {
    520: "Possible annotation issue",
    529: "Related information, incomplete answer",
    530: "Relevant evidence missed",
    550: "Relevant evidence missed",
    35: "Partial or indirect evidence",
    112: "Relevant evidence missed",
    191: "Related information, incomplete answer",
    341: "Question-context mismatch",
    814: "Relevant evidence missed",
    867: "Partial or indirect evidence",
    1058: "Related information, incomplete answer",
    1059: "Question-context mismatch",
}

sample_errors = sample_errors.copy()
sample_errors["category"] = sample_errors.index.map(category_map)

category_counts = (
    sample_errors["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

display(category_counts)


,category,count
0,Partial or indirect evidence,2
1,Relevant evidence missed,2
2,"Related information, incomplete answer",2
3,Question-context mismatch,1
4,Possible annotation issue,1


In [24]:
display(
    sample_errors[
        ["lang", "error_type", "category"]
    ].sort_values(["lang", "error_type"])
)

,lang,error_type,category
35,ar,false_negative,Partial or indirect evidence
112,ar,false_negative,Relevant evidence missed
191,ar,false_positive,"Related information, incomplete answer"
341,ar,false_positive,Question-context mismatch
569,ko,false_negative,NaN
644,ko,false_negative,NaN
468,ko,false_positive,NaN
520,ko,false_positive,Possible annotation issue
814,te,false_negative,Relevant evidence missed
867,te,false_negative,Partial or indirect evidence


In [25]:
display(
    pd.crosstab(
        sample_errors["category"],
        sample_errors["lang"]
    )
)


lang,ar,ko,te
category,,,
Partial or indirect evidence,1,0,1
Possible annotation issue,0,1,0
Question-context mismatch,1,0,0
"Related information, incomplete answer",1,0,1
Relevant evidence missed,1,0,1


In [ ]:
out_dir = Path("../results/qa_analysis")
out_dir.mkdir(parents=True, exist_ok=True)

n = len(df_val_tokenised)
ans_export = df_val_tokenised[["lang", "question", "context", "answerable", "answer"]].copy().reset_index(drop=True)
ans_export = ans_export.rename(columns={"answerable": "gold_answerable", "answer": "gold_answer"})
ans_export.insert(0, "example_index", np.arange(n))
ans_export["gold_answerable"] = ans_export["gold_answerable"].astype(int)
ans_export["lr_pred_answerable"] = np.asarray(lr_pred).astype(int)
ans_export["gru_pred_answerable"] = np.asarray(gru_pred).astype(int)
ans_export["gru_pred_prob"] = np.asarray(gru_prob)

export_path = out_dir / "week39_answerability_validation.csv"
ans_export.to_csv(export_path, index=False)
print(f"Saved {len(ans_export)} answerability predictions to {export_path}")
print("LR macro-F1:", round(f1_score(y_val, lr_pred, average="macro"), 4))
print("GRU macro-F1:", round(f1_score(y_val, gru_pred, average="macro"), 4))

Saved 1155 answerability predictions to ../results/qa_analysis/week39_answerability_validation.csv
LR macro-F1: 0.7236
GRU macro-F1: 0.8612


## Manual error review of the pretrained GRU (Week 39)

The task asks for at least ten validation errors, with false positives and false negatives from all three languages,
failure categories, category counts and representative examples. Terminology used below:

* **false negative (FN)**: the question is answerable, the GRU predicts unanswerable;
* **false positive (FP)**: the question is unanswerable, the GRU predicts answerable.

The review sample contains **all** FNs and a seeded random subsample of the FPs per language (Telugu has 48 FPs, so
it is subsampled). Category counts are therefore counts *within the reviewed sample*, not within all errors; the
full error counts per language are printed alongside.

In [ ]:
# error_review: build the sample (seeded, validation data only)
REVIEW_SEED = 42
FP_PER_LANG = {"ar": 3, "ko": 4, "te": 12}   # all FPs for ar/ko, a subsample for te

review_src = df_val_tokenised.reset_index(drop=True).copy()
review_src["example_index"] = np.arange(len(review_src))
review_src["gold"] = np.asarray(y_val)
review_src["gru_pred"] = np.asarray(gru_pred).astype(int)
review_src["gru_prob_answerable"] = np.asarray(gru_prob)
review_src["lr_pred"] = np.asarray(lr_pred).astype(int)
review_src["context_words"] = review_src["context"].str.split().str.len()

fn_all = review_src[(review_src["gold"] == 1) & (review_src["gru_pred"] == 0)].assign(error_type="false_negative")
fp_all = review_src[(review_src["gold"] == 0) & (review_src["gru_pred"] == 1)].assign(error_type="false_positive")

fp_sample = pd.concat([
    g.sample(n=min(len(g), FP_PER_LANG[lang]), random_state=REVIEW_SEED)
    for lang, g in fp_all.groupby("lang")
])
error_review = (
    pd.concat([fn_all, fp_sample])
    .sort_values(["error_type", "lang", "example_index"])
    .reset_index(drop=True)
)

print("All GRU errors on validation:")
display(pd.crosstab(pd.concat([fn_all, fp_all])["lang"], pd.concat([fn_all, fp_all])["error_type"]))
print("Reviewed sample:")
display(pd.crosstab(error_review["lang"], error_review["error_type"]))

In [ ]:
# error_review: print every sampled error so it can be read and labelled by hand
for _, r in error_review.iterrows():
    print("=" * 100)
    print(f"example_index={r.example_index} | {r.lang} | {r.error_type} | "
          f"GRU P(answerable)={r.gru_prob_answerable:.3f} | LR pred={r.lr_pred} | context words={r.context_words}")
    print("Q:", r.question)
    if r.gold == 1:
        print("GOLD ANSWER:", r.answer)
    print("C:", r.context[:700] + (" ..." if len(r.context) > 700 else ""))

In [ ]:
# error_review: also save the sample (empty `category` column) in case you prefer to label it in a spreadsheet
review_path = Path("../results/qa_analysis/week39_error_review.csv")
review_path.parent.mkdir(parents=True, exist_ok=True)
(error_review[["example_index", "lang", "error_type", "gru_prob_answerable", "question", "answer", "context"]]
    .assign(category="")
    .to_csv(review_path, index=False))
print("Saved", len(error_review), "errors to", review_path)

### Proposed error taxonomy

Task-motivated categories (edit after reading the examples; every category used must be defined here):

| Category | Applies to | Meaning |
|---|---|---|
| `yes_no_question` | FN | Boolean question (e.g. Arabic *هل*, Korean *-가?*) whose answer is yes/no and not a span; the LR learned a strong negative weight for *هل* |
| `indirect_evidence` | FN | The answer is in the context but needs paraphrase, inference or numeric reasoning |
| `evidence_missed` | FN | The context states the answer explicitly but the classifier does not detect it (surface mismatch across languages) |
| `related_but_missing` | FP | The context is on the same topic or entity but never states the requested fact |
| `question_context_mismatch` | FP | The context is about a different entity or topic from the question |
| `annotation_issue` | FN/FP | The gold label looks wrong or debatable after reading the context |

In [ ]:
# error_review: fill in ONE category per reviewed example after reading the printout above.
# Keys are example_index values from `error_review`; values must come from TAXONOMY.
TAXONOMY = {
    "yes_no_question", "indirect_evidence", "evidence_missed",
    "related_but_missing", "question_context_mismatch", "annotation_issue",
}
ERROR_CATEGORIES = {
    # example_index: "category",
}

In [ ]:
# error_review: category counts and representative examples (runs once ERROR_CATEGORIES is filled in)
labelled = error_review[error_review["example_index"].isin(ERROR_CATEGORIES)].copy()
labelled["category"] = labelled["example_index"].map(ERROR_CATEGORIES)

unknown = set(ERROR_CATEGORIES.values()) - TAXONOMY
assert not unknown, f"categories not in TAXONOMY: {unknown}"
missing = set(error_review["example_index"]) - set(ERROR_CATEGORIES)
assert not missing, f"{len(missing)} reviewed errors still unlabelled: {sorted(missing)}"
assert len(labelled) >= 10 and set(labelled["lang"]) == {"ar", "ko", "te"}
assert set(labelled["error_type"]) == {"false_negative", "false_positive"}

print("Category counts by error type:")
display(pd.crosstab(labelled["category"], labelled["error_type"], margins=True))
print("Category counts by language:")
display(pd.crosstab(labelled["category"], labelled["lang"], margins=True))

print("One representative example per category:")
display(
    labelled.sort_values("example_index")
    .groupby("category")
    .head(1)[["category", "lang", "error_type", "example_index", "question", "answer"]]
)
labelled[["example_index", "lang", "error_type", "category"]].to_csv(
    "../results/qa_analysis/week39_error_categories.csv", index=False
)